# 3 · Efficient Frontier

Monte Carlo simulation: 20,000 random portfolios → Return / Volatility / Sharpe cloud.  
Output feeds the visualisation notebook.

In [1]:
import pandas as pd
import numpy as np
from scipy.optimize import minimize
from pathlib import Path
import sys
sys.path.insert(0, '..')
from utils import (
    ANALYSIS_DIR, RF_MONTHLY, PERIODS_PER_YEAR,
    portfolio_return, portfolio_volatility,
    annualise_return, annualise_vol
)

analysis_path = Path(ANALYSIS_DIR)

returns = pd.read_csv(
    analysis_path / "india_equity_returns.csv",
    index_col="Date", parse_dates=True
)

mu  = returns.mean().values
cov = returns.cov().values
n   = len(mu)

N_SIMS = 20_000
np.random.seed(0)

records = []
for _ in range(N_SIMS):
    w = np.random.dirichlet(np.ones(n))
    r = portfolio_return(w, mu)
    v = portfolio_volatility(w, cov)
    s = (r - RF_MONTHLY) / v
    records.append({
        "Return"         : r,
        "Volatility"     : v,
        "Sharpe"         : s,
        "Annual_Return"  : annualise_return(r),
        "Annual_Vol"     : annualise_vol(v),
        "Annual_Sharpe"  : s * np.sqrt(PERIODS_PER_YEAR),
    })

sim = pd.DataFrame(records)

# ── True frontier: solve for min-vol at each target return ───────────────────
target_returns = np.linspace(mu.min(), mu.max(), 80)
frontier = []

for target in target_returns:
    constraints = [
        {"type": "eq", "fun": lambda w: np.sum(w) - 1},
        {"type": "eq", "fun": lambda w, t=target: portfolio_return(w, mu) - t},
    ]
    bounds = tuple((0.0, 1.0) for _ in range(n))
    res = minimize(
        lambda w: portfolio_volatility(w, cov),
        np.full(n, 1/n),
        method="SLSQP",
        bounds=bounds,
        constraints=constraints,
        options={"ftol": 1e-12, "maxiter": 1000}
    )
    if res.success:
        frontier.append({
            "Return"     : target,
            "Volatility" : res.fun,
            "Sharpe"     : (target - RF_MONTHLY) / res.fun,
        })

frontier_df = pd.DataFrame(frontier)

sim.to_csv(analysis_path / "portfolio_simulations.csv", index=False)
frontier_df.to_csv(analysis_path / "efficient_frontier.csv", index=False)

print(f"Simulations : {len(sim):,}")
print(f"Frontier pts: {len(frontier_df)}")
print(f"Max Sharpe  : {sim['Sharpe'].max():.4f}  (monthly)")
print(f"             {sim['Annual_Sharpe'].max():.4f}  (annualised ×√12)")
print("\nSaved to analysis/")

Simulations : 20,000
Frontier pts: 80
Max Sharpe  : 0.3171  (monthly)
             1.0985  (annualised ×√12)

Saved to analysis/
